# 🕷️ Notebook 01: ทดลอง Web Scraping เว็บไซต์ Yuedpao.com

โน้ตบุ๊กนี้ใช้สำหรับ POC (Proof of Concept) การดึงข้อมูลจากเว็บไซต์ **Yuedpao** เพื่อสร้างฐานข้อมูลสินค้า และ Domain Vocabulary สำหรับระบบ LINE Chatbot

In [ ]:
# 1. Import Libraries ที่จำเป็น
import requests
from bs4 import BeautifulSoup
import json
import time
import xml.etree.ElementTree as ET

HEADERS = {
    'User-Agent': 'YuedpaoBot-Scraper/1.0 (Educational/Bot Integration Project)'
}
BASE_URL = 'https://www.yuedpao.com'
print('Libraries imported successfully!')

## 📌 Step 1: ดึง URL รายการสินค้าจาก Sitemap.xml

In [ ]:
sitemap_url = f'{BASE_URL}/sitemap.xml'
response = requests.get(sitemap_url, headers=HEADERS)
print(f'Sitemap Status Code: {response.status_code}')

if response.status_code == 200:
    root = ET.fromstring(response.content)
    urls = [elem.text for elem in root.iter('{http://www.sitemaps.org/schemas/sitemap/0.9}loc')]
    print(f'พบ URL ทั้งหมดใน Sitemap: {len(urls)} รายการ')
    # แสดงตัวอย่าง 5 URL แรก
    for u in urls[:5]:
        print('-', u)
else:
    print('ไม่สามารถดึง Sitemap ได้')

## 📌 Step 2: ทดลองดึงข้อมูลรายละเอียดสินค้า 1 หน้า (Single Product Scraping Test)

In [ ]:
# ทดลองดึงข้อมูลจากหน้าสินค้าตัวอย่าง
test_product_url = f'{BASE_URL}/'
res = requests.get(test_product_url, headers=HEADERS)
soup = BeautifulSoup(res.text, 'html.parser')

print(f'Page Title: {soup.title.string if soup.title else "N/A"}')

# ตัวอย่างโครงสร้างดึงข้อมูลสินค้า
sample_product = {
    'product_id': 'yp-001',
    'sku': 'YP-ULTRASOFT-NAVY',
    'name': 'เสื้อยืด Oversize Ultrasoft',
    'category': 'เสื้อยืด',
    'fabric_collection': 'Ultrasoft',
    'price': 390,
    'discount_price': None,
    'colors': ['Classic Navy', 'White', 'Black'],
    'sizes': ['S', 'M', 'L', 'XL', '2XL'],
    'is_in_stock': True,
    'primary_image_url': 'https://www.yuedpao.com/images/ultrasoft_navy.jpg',
    'checkout_url': test_product_url
}

print('Sample Product Data Struct:')
print(json.dumps(sample_product, indent=2, ensure_ascii=False))

## 📌 Step 3: ทดลองสร้าง Domain Vocabulary สำหรับ NLP Edit Distance

In [ ]:
domain_vocab = {
    'brand_colors': ['Amber Wood', 'Shadow Gray', 'Salmon Rose', 'Cha Thai', 'Classic Navy', 'Dark Moss', 'Coffee Brown'],
    'product_styles': ['Oversize', 'Crop', 'Unisex', 'Cargo', 'Boxer Briefs', 'Crewneck', 'V-Neck'],
    'fabric_technologies': ['Non-iron', 'Ultrasoft', 'Tailor Cool', 'MotionSkin', 'Ultra Flow', 'Feather Comfort'],
    'apparel_types': ['เสื้อยืด', 'เสื้อเชิ้ต', 'โปโล', 'กางเกงยีนส์', 'ชุดกีฬา', 'กางเกงใน']
}

with open('domain_vocabulary.json', 'w', encoding='utf-8') as f:
    json.dump(domain_vocab, f, indent=2, ensure_ascii=False)

print('บันทึก domain_vocabulary.json สำเร็จ!')